# Bechert 1997 Blade-Riblet Replication (Stage 5)

**Streamwise-periodic channel** (see `STAGE-5-REDESIGN.md`). The riblet
domain is a periodic channel normalized on the half-height delta=1,
u_tau=1, nu=1/Re_tau. Riblet pitch s = s+/Re_tau.

Drag reduction is measured from the mean pressure gradient that
`meanVelocityForce` applies to hold the bulk velocity Ubar:

    DR% = (dpdx_smooth - dpdx_riblet) / dpdx_smooth * 100

The smooth baseline is s+-independent (no riblets; the channel is
y-homogeneous and the smooth mesh has identical cell count for every
s+), so one smooth run serves all sweep points.

Calibration target: Bechert et al. 1997 JFM 338:59-87 Fig 5, blade
riblet h/s=0.5 — peak DR ~9.9% near s+~17, crossover s+~27.
Hypothesis bound: peak within +/-2 pp, crossover within +/-3.

In [ ]:
from __future__ import annotations
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path('/opt/aero-research-platform')
RESULTS = REPO / 'results' / '02-flat-plate-riblet-bechert'
BECHERT_CSV = REPO / 'data' / 'bechert-1997-fig5' / 'digitized.csv'
S_PLUS_SWEEP = [5, 10, 15, 17, 20, 25, 30, 35, 40]
TAIL = 1000          # iterations averaged for the converged dp/dx
PEAK_DR_TARGET = 9.9
PEAK_DR_TOL_PP = 2.0
PEAK_S_PLUS = 17
CROSSOVER_S_PLUS = 27
CROSSOVER_TOL = 3

In [ ]:
def mean_dpdx(log_simplefoam: Path, tail: int = TAIL) -> float:
    """Mean applied pressure gradient over the last `tail` iterations.

    Parses 'Pressure gradient source: ... pressure gradient = X' lines
    emitted by the meanVelocityForce fvOption each iteration.
    """
    if not log_simplefoam.exists():
        return float('nan')
    vals = []
    pat = re.compile(r'pressure gradient = ([-\d.eE+]+)')
    for line in log_simplefoam.read_text(errors='ignore').splitlines():
        m = pat.search(line)
        if m:
            vals.append(float(m.group(1)))
    if not vals:
        return float('nan')
    return float(np.mean(vals[-tail:]))


dpdx_smooth = mean_dpdx(RESULTS / 'smooth' / 'log.simpleFoam')
print(f'smooth baseline dp/dx = {dpdx_smooth:.5f}')

In [ ]:
rows = []
for s_plus in S_PLUS_SWEEP:
    dpdx_r = mean_dpdx(RESULTS / f'riblet-{s_plus}' / 'log.simpleFoam')
    if np.isfinite(dpdx_r) and np.isfinite(dpdx_smooth) and dpdx_smooth != 0:
        dr = (dpdx_smooth - dpdx_r) / dpdx_smooth * 100.0
    else:
        dr = float('nan')
    rows.append({'s_plus': s_plus, 'dpdx_riblet': dpdx_r,
                 'dpdx_smooth': dpdx_smooth, 'dr_percent': dr})
measured = pd.DataFrame(rows)
measured

In [ ]:
bechert = pd.read_csv(BECHERT_CSV, comment='#')
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bechert['s_plus'], bechert['dr_percent'], 'o-', color='black',
        label='Bechert 1997 Fig 5 (digitized)')
ax.plot(measured['s_plus'], measured['dr_percent'], 's-', color='C3',
        label='Stage 5 — RANS k-omega SST (this work)')
ax.axhline(0, color='grey', lw=0.5)
ax.axhspan(PEAK_DR_TARGET - PEAK_DR_TOL_PP, PEAK_DR_TARGET + PEAK_DR_TOL_PP,
           color='lightyellow', alpha=0.6, label=f'+/-{PEAK_DR_TOL_PP} pp Bechert-peak band')
ax.set_xlabel('s+ = s u_tau / nu')
ax.set_ylabel('Drag reduction (%)')
ax.set_title('Bechert 1997 blade-riblet replication, h/s=0.5')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# PASS / PARTIAL / FAIL against the pre-registered hypothesis bound.
valid = measured.dropna(subset=['dr_percent'])
if valid.empty:
    verdict, peak_dr, s_peak, crossover = 'NO_DATA', float('nan'), float('nan'), float('nan')
else:
    pi = valid['dr_percent'].idxmax()
    peak_dr = float(valid.loc[pi, 'dr_percent'])
    s_peak = float(valid.loc[pi, 's_plus'])
    crossover = float('nan')
    sv = valid.sort_values('s_plus')
    for i in range(len(sv) - 1):
        a, b = sv.iloc[i], sv.iloc[i + 1]
        if a['dr_percent'] > 0 >= b['dr_percent']:
            frac = a['dr_percent'] / (a['dr_percent'] - b['dr_percent'])
            crossover = float(a['s_plus'] + frac * (b['s_plus'] - a['s_plus']))
            break
    within_peak = (abs(peak_dr - PEAK_DR_TARGET) <= PEAK_DR_TOL_PP
                   and abs(s_peak - PEAK_S_PLUS) <= CROSSOVER_TOL)
    within_cross = np.isfinite(crossover) and abs(crossover - CROSSOVER_S_PLUS) <= CROSSOVER_TOL
    verdict = 'PASS' if within_peak and within_cross else 'PARTIAL' if within_peak or within_cross else 'FAIL'

print(f'measured peak DR  : {peak_dr:.2f}%  at s+ = {s_peak:.0f}')
print(f'Bechert peak DR   : {PEAK_DR_TARGET}%  at s+ ~ {PEAK_S_PLUS}')
print(f'measured crossover: s+ = {crossover:.1f}   (Bechert ~{CROSSOVER_S_PLUS})')
print(f'VERDICT           : {verdict}')
RESULTS.mkdir(parents=True, exist_ok=True)
measured.to_csv(RESULTS / 'dr_vs_s_plus.csv', index=False)
print(f'wrote {RESULTS / "dr_vs_s_plus.csv"}')